# Tạo bộ ảnh tiền Việt Nam từ ZIP gốc

Notebook đọc ảnh và nhãn YOLO trong ZIP, chọn ảnh đại diện đa dạng, áp dụng các quyết định review đã chốt, rồi cắt và lưu ảnh train.

Chạy lần lượt từ trên xuống. Kết quả nằm ngoài thư mục `DATA`.

## 1. Chuẩn bị thư viện và thư mục kết quả

Chọn ZIP gốc và tạo thư mục riêng cho lần chạy. Các CSV trong thư mục kết quả là báo cáo notebook sinh ra.

In [1]:
from pathlib import Path
import shutil
from zipfile import ZipFile, ZIP_DEFLATED
from collections import Counter
from datetime import datetime
from PIL import Image, ImageOps
from IPython.display import display
import pandas as pd
import numpy as np
import cv2
import yaml
import hashlib
import math
import re
import io
import json
import matplotlib.pyplot as plt


In [2]:
WORK = Path.cwd()
RAW_ZIP = Path.home() / "Downloads" / "Vietnamese Currency.v6i.yolov5pytorch.zip"
OUTPUT = Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT.mkdir(parents=True, exist_ok=True)
TARGET_CLASSES = ["10000", "20000", "50000", "100000", "200000", "500000"]
EXPECTED_COUNTS = {"10000": 19, "20000": 18, "50000": 9,
                   "100000": 9, "200000": 19, "500000": 18}
assert RAW_ZIP.is_file(), f"Không tìm thấy ZIP: {RAW_ZIP}"
archive = ZipFile(RAW_ZIP)
members = set(archive.namelist())
print("ZIP:", RAW_ZIP, "\nKết quả:", OUTPUT)

ZIP: Downloads/Vietnamese Currency.v6i.yolov5pytorch.zip 
Kết quả: C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6\20261025_183559_632069


## 2. Xem ảnh và nhãn trong ZIP

`data.yaml` ánh xạ class ID sang mệnh giá. Chỉ ảnh có đúng một bbox YOLO hợp lệ thuộc sáu mệnh giá mục tiêu mới vào pool; các lớp 1.000, 2.000 và 5.000 đồng bị loại.

In [3]:
config = yaml.safe_load(archive.read("data.yaml").decode("utf-8-sig"))
class_names = config["names"]
class_to_vnd = {class_id: str(class_names[class_id]) for class_id in range(len(class_names))}
class_rows = [{"class_id": class_id, "tên trong ZIP": class_names[class_id],
               "mệnh giá": class_to_vnd[class_id]} for class_id in class_to_vnd]
display(pd.DataFrame(class_rows))
image_members = []
for name in sorted(members):
    is_image = "/images/" in name
    is_picture = Path(name).suffix.lower() in [".jpg", ".jpeg", ".png"]
    if is_image and is_picture:
        image_members.append(name)
def label_for(image_member):
    label_name = image_member.replace("/images/", "/labels/")
    return label_name.rsplit(".", 1)[0] + ".txt"
missing_labels = [name for name in image_members if label_for(name) not in members]
original_splits = Counter(name.split("/images/")[0].split("/")[-1] for name in image_members)
print("Ảnh trong images:", len(image_members), "| Split gốc:", dict(original_splits))
print("Có label:", len(image_members) - len(missing_labels), "| Thiếu label:", len(missing_labels))
example = next((name for name in image_members if label_for(name) in members), "")
print("Ví dụ ảnh:", example)
if example:
    print("YOLO label:", archive.read(label_for(example)).decode().strip())

,class_id,tên trong ZIP,mệnh giá
0,0,1000,1000
1,1,10000,10000
2,2,100000,100000
3,3,2000,2000
4,4,20000,20000
5,5,200000,200000
6,6,5000,5000
7,7,50000,50000
8,8,500000,500000


Ảnh trong images: 5262 | Split gốc: {'test': 218, 'train': 4605, 'valid': 439}
Có label: 5262 | Thiếu label: 0
Ví dụ ảnh: test/images/1101_jpg.rf.51ea8d7c33cfc761f0cbfecb1430913c.jpg
YOLO label: 8 0.4140625 0.325 0.1609375 0.265625
7 0.646875 0.4328125 0.2125 0.275
2 0.453125 0.684375 0.1953125 0.325


## 3. Đọc nhãn và gộp tên ảnh nguồn

Các file cùng stem sau khi bỏ hậu tố `.rf.<hash>` và hậu tố `_jpg`, `_png` hoặc `_jpeg` được coi là một nhóm nguồn. Ưu tiên ảnh có nhãn hợp lệ, bản không phải biến thể `.rf` nếu có, kích thước rộng hơn, rồi tên đường dẫn Windows tăng dần.

In [4]:
def source_name(member):
    name = re.sub(r"\.rf\.[0-9a-f]+$", "", Path(member).stem, flags=re.I)
    return re.sub(r"_(jpg|png|jpeg)$", "", name, flags=re.I)
def read_boxes(label_member):
    if label_member not in members:
        return []
    boxes = []
    for line in archive.read(label_member).decode("utf-8-sig").splitlines():
        values = line.split()
        if len(values) != 5:
            return []
        try:
            class_id = int(values[0])
            x, y, w, h = map(float, values[1:])
        except ValueError:
            return []
        if (class_id not in class_to_vnd or not all(math.isfinite(v) for v in (x, y, w, h))
                or not 0 <= x <= 1 or not 0 <= y <= 1 or not 0 < w <= 1 or not 0 < h <= 1):
            return []
        boxes.append({"raw_label": class_to_vnd[class_id], "cx": x, "cy": y, "bw": w, "bh": h})
    return boxes
records = []
for member in image_members:
    boxes = read_boxes(label_for(member))
    try:
        with Image.open(io.BytesIO(archive.read(member))) as image:
            image.load()
            width, height = image.size
        readable = True
    except Exception:
        width, height, readable = 0, 0, False
    valid = readable and len(boxes) == 1 and boxes[0]["raw_label"] in TARGET_CLASSES
    row = {"member": member, "path_order": member.replace("/", "\\"),
           "source_name": source_name(member), "annotation_member": label_for(member),
           "width": width, "height": height, "valid": valid,
           "is_variant": bool(re.search(r"\.rf\.[0-9a-f]+$", Path(member).stem, flags=re.I))}
    row.update(boxes[0] if len(boxes) == 1 else
               {"raw_label": "", "cx": 0, "cy": 0, "bw": 0, "bh": 0})
    records.append(row)
all_images = pd.DataFrame(records)
all_images = all_images.sort_values(["valid", "is_variant", "width", "path_order"],
                                    ascending=[False, True, False, True])
representatives = all_images.drop_duplicates("source_name")
pool = representatives[representatives["valid"]].copy().sort_values("path_order").reset_index(drop=True)
print("File ảnh:", len(all_images), "→ nhóm tên nguồn:", len(representatives))
print("Nhóm hợp lệ:", len(pool), "| Nhóm thiếu/không đạt annotation:", len(representatives) - len(pool))
all_images.to_csv(OUTPUT / "01_ALL_RAW_IMAGES.csv", index=False)
display(pool.groupby("raw_label").size().reindex(TARGET_CLASSES).to_frame("ảnh đại diện hợp lệ"))

File ảnh: 5262 → nhóm tên nguồn: 2043
Nhóm hợp lệ: 280 | Nhóm thiếu/không đạt annotation: 1763


,ảnh đại diện hợp lệ
raw_label,
10000,60
20000,58
50000,14
100000,10
200000,66
500000,72


## 4. Chọn ảnh đa dạng trong từng mệnh giá

Mỗi ảnh có 9 đặc trưng từ ảnh gốc: diện tích bbox, rộng/cao bbox, tâm x/y, độ sáng, độ tương phản, độ phân bố sắc độ và tỷ lệ cạnh. Đặc trưng chỉ dùng để lấy mẫu.

Nhóm ảnh tiền này dùng stem nguồn làm nhóm chụp. Bộ chọn giới hạn tối đa 20 ảnh mỗi nhóm và chọn lần lượt ảnh xa nhất so với các ảnh đã lấy trên các đặc trưng đã chuẩn hóa về 0–1.

In [5]:
def capture_group(member):
    return source_name(member)
pool["capture_group"] = pool["member"].map(capture_group)
summaries = []
for row in pool.to_dict("records"):
    with Image.open(io.BytesIO(archive.read(row["member"]))) as image:
        gray = np.asarray(ImageOps.grayscale(image.resize((96, 96))), dtype=np.uint8)
    histogram = cv2.calcHist([gray], [0], None, [32], [0, 256]).ravel()
    summaries.append({"box_area": row["bw"] * row["bh"], "brightness": float(gray.mean()),
                      "contrast": float(gray.std()),
                      "tone_detail": float(histogram.dot(-np.log2((histogram + 1e-9) / gray.size))),
                      "edge_fraction": float(np.mean(cv2.Canny(gray, 80, 160) > 0))})
pool = pd.concat([pool, pd.DataFrame(summaries)], axis=1)
pool.to_csv(OUTPUT / "02_CANDIDATES_FROM_RAW.csv", index=False)
pool = pd.read_csv(OUTPUT / "02_CANDIDATES_FROM_RAW.csv", keep_default_na=False,
                   dtype={"raw_label": str, "source_name": str, "capture_group": str})
feature_columns = ["box_area", "bw", "bh", "cx", "cy", "brightness", "contrast", "tone_detail", "edge_fraction"]
def choose_different_images(group, limit=20, per_capture=20):
    group = group.sort_values("path_order").reset_index(drop=True)
    values = group[feature_columns].to_numpy(float)
    values = (values - values.min(axis=0)) / np.maximum(values.max(axis=0) - values.min(axis=0), 1e-9)
    chosen = [0]
    distances = np.sum((values - values[0]) ** 2, axis=1)
    capture_counts = Counter([group.iloc[0]["capture_group"]])
    while len(chosen) < min(limit, len(group)):
        scores = distances.copy()
        for i, capture in enumerate(group["capture_group"]):
            if capture_counts[capture] >= per_capture:
                scores[i] = -1
        scores[chosen] = -1
        if scores.max() < 0:
            break
        next_image = int(scores.argmax())
        chosen.append(next_image)
        capture_counts[group.iloc[next_image]["capture_group"]] += 1
        distance_to_new_image = np.sum((values - values[next_image]) ** 2, axis=1)
        distances = np.minimum(distances, distance_to_new_image)
    return group.iloc[chosen].copy()
sampled_groups = []
for label in TARGET_CLASSES:
    chosen_for_label = choose_different_images(pool[pool["raw_label"] == label])
    sampled_groups.append(chosen_for_label)
sampled = pd.concat(sampled_groups, ignore_index=True)
print("Ảnh được lấy mẫu trước lọc gần trùng:", len(sampled))
display(sampled.groupby("raw_label").size().reindex(TARGET_CLASSES).to_frame("ảnh lấy mẫu"))

Ảnh được lấy mẫu trước lọc gần trùng: 104


,ảnh lấy mẫu
raw_label,
10000,20
20000,20
50000,14
100000,10
200000,20
500000,20


## 5. Lọc gần trùng và áp dụng review

Tính dHash và pHash trên ảnh xám ở bốn hướng xoay cho toàn bộ pool. Hai ảnh cùng dHash hoặc có pHash cách nhau tối đa 4 bit được gom chung bằng union-find; trong các ảnh đã lấy mẫu, giữ đường dẫn sắp xếp trước.

Các quyết định nguồn V2.1, review người và visual QA được ghi rõ theo stem. Chúng là quyết định loại/đổi nhãn cụ thể, không phải danh sách cho phép ảnh dương tính.

In [6]:
def rotation_hashes(member):
    data = np.frombuffer(archive.read(member), dtype=np.uint8)
    gray = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    gray = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)
    dhashes, phashes = [], []
    for rotation in range(4):
        rotated = np.rot90(gray, rotation).copy()
        small = cv2.resize(rotated, (9, 8), interpolation=cv2.INTER_AREA)
        dhashes.append(sum(int(small[y, x] > small[y, x + 1]) << (y * 8 + x)
                           for y in range(8) for x in range(8)))
        small32 = cv2.resize(rotated, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
        frequencies = cv2.dct(small32)[:8, :8].reshape(-1)[1:]
        middle = np.median(frequencies)
        phashes.append(sum(int(value > middle) << i for i, value in enumerate(frequencies)))
    return min(dhashes), min(phashes)
image_hashes = [rotation_hashes(member) for member in pool["member"]]
parent = list(range(len(pool)))
def find_group(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def join_groups(i, j):
    left, right = find_group(i), find_group(j)
    if left != right:
        parent[right] = left
first_dhash = {}
for i, (dhash, phash) in enumerate(image_hashes):
    if dhash in first_dhash:
        join_groups(i, first_dhash[dhash])
    else:
        first_dhash[dhash] = i
    for j in range(i):
        earlier_phash = image_hashes[j][1]
        if (phash ^ earlier_phash).bit_count() <= 4:
            join_groups(i, j)
group_by_member = {member: find_group(i) for i, member in enumerate(pool["member"])}
sampled["duplicate_group"] = sampled["member"].map(group_by_member)
unique_samples = sampled.sort_values("path_order").drop_duplicates("duplicate_group").copy()
duplicate_rows = sampled[~sampled["member"].isin(unique_samples["member"])].copy()
duplicate_rows["reason"] = "Cùng nhóm hash xoay với ảnh đại diện khác đã chọn"
duplicate_rows.to_csv(OUTPUT / "03_AUTOMATIC_DUPLICATE_REMOVALS.csv", index=False)
HISTORICAL_REMOVE = {
    "frame0-00-02-33-4-": "Cùng chuỗi ảnh 50.000 đồng; giữ ảnh đại diện rõ hơn theo redundancy review V2.1.",
    "frame0-00-04-53": "Cùng chuỗi ảnh 50.000 đồng; giữ ảnh đại diện rõ hơn theo redundancy review V2.1.",
}
CROSS_SOURCE_REMOVE = {
    "34b4be1f-307c-4fbd-a188-e0b50ae0a1df": "Trùng mã stem nguồn UUID với nhóm currency trong blindtrix_v1; giữ một nguồn đại diện.",
    "583b48a7-7ed6-4e18-a0ef-d934536bb51b": "Trùng mã stem nguồn UUID với nhóm currency trong blindtrix_v1; giữ một nguồn đại diện.",
    "b6efc05b-46a0-49c1-9e33-146a5274a151": "Trùng mã stem nguồn UUID với nhóm currency trong blindtrix_v1; giữ một nguồn đại diện.",
}
HUMAN_REMOVE = {
    "z5422517290884_57c58760a1304bcd7773ce3c7eb6dbfa-1-": "Review người: có quá nhiều tờ tiền, không phù hợp mục tiêu định hướng.",
}
VISUAL_REMOVE = {
    "18d68b3c-3fe0-41d1-9f2a-deba4138fa07": "Visual QA: crop chỉ giữ phần quá nhỏ của tờ tiền.",
}
remove_reason = {**HISTORICAL_REMOVE, **CROSS_SOURCE_REMOVE, **HUMAN_REMOVE, **VISUAL_REMOVE}
review_removed = unique_samples[unique_samples["source_name"].isin(remove_reason)].copy()
review_removed["reason"] = review_removed["source_name"].map(remove_reason)
review_removed.to_csv(OUTPUT / "04_REVIEW_REMOVALS.csv", index=False)
selected = unique_samples[~unique_samples["source_name"].isin(remove_reason)].copy()
print("Lấy mẫu:", len(sampled), "| Sau lọc hash:", len(unique_samples),
      "| Loại theo quyết định review:", len(review_removed), "| Còn lại:", len(selected))
print("Stem review tìm thấy:", sorted(review_removed["source_name"].tolist()))

Lấy mẫu: 104 | Sau lọc hash: 100 | Loại theo quyết định review: 7 | Còn lại: 93
Stem review tìm thấy: ['18d68b3c-3fe0-41d1-9f2a-deba4138fa07', '34b4be1f-307c-4fbd-a188-e0b50ae0a1df', '583b48a7-7ed6-4e18-a0ef-d934536bb51b', 'b6efc05b-46a0-49c1-9e33-146a5274a151', 'frame0-00-02-33-4-', 'frame0-00-04-53', 'z5422517290884_57c58760a1304bcd7773ce3c7eb6dbfa-1-']


## 6. Tính bbox pixel và crop có nền

Đọc lại nhãn YOLO gốc (không lấy số từ CSV feature). Cạnh trái/trên dùng `floor`, phải/dưới dùng `ceil`, sau đó chặn trong ảnh. Chỉ giữ bbox có cạnh ngắn ít nhất 120 px; crop thêm 20% bbox mỗi phía.

In [7]:
def pixel_bbox(row):
    width, height = int(row["width"]), int(row["height"])
    box = read_boxes(row["annotation_member"])[0]
    left, top = box["cx"] - box["bw"] / 2, box["cy"] - box["bh"] / 2
    right, bottom = box["cx"] + box["bw"] / 2, box["cy"] + box["bh"] / 2
    return (max(0, math.floor(left * width)), max(0, math.floor(top * height)),
            min(width, math.ceil(right * width)), min(height, math.ceil(bottom * height)))
def context_box(box, width, height):
    x1, y1, x2, y2 = box
    box_width, box_height = x2 - x1, y2 - y1
    return (max(0, round(x1 - 0.20 * box_width)), max(0, round(y1 - 0.20 * box_height)),
            min(width, round(x2 + 0.20 * box_width)), min(height, round(y2 + 0.20 * box_height)))
bboxes, bbox_short_sides = [], []
for row in selected.to_dict("records"):
    bbox = pixel_bbox(row)
    bboxes.append(bbox)
    bbox_short_sides.append(min(bbox[2] - bbox[0], bbox[3] - bbox[1]))
selected["bbox"], selected["bbox_short_side_px"] = bboxes, bbox_short_sides
print("Bbox đạt ngưỡng 120 px:", sum(side >= 120 for side in bbox_short_sides), "/", len(selected))
selected = selected[selected["bbox_short_side_px"] >= 120].copy()
display(selected.groupby("raw_label").size().reindex(TARGET_CLASSES).to_frame("ảnh sau ngưỡng bbox"))

Bbox đạt ngưỡng 120 px: 92 / 93


,ảnh sau ngưỡng bbox
raw_label,
10000,19
20000,18
50000,9
100000,9
200000,19
500000,18


## 7. Lưu crop PNG và manifest

Mọi ảnh đầu ra thuộc split `train`. Manifest ghi nguồn ảnh, nhãn gốc/cuối, bbox YOLO, tọa độ crop và SHA-256 của ảnh nguồn lẫn PNG cuối.

In [8]:
DATA_OUT = OUTPUT / "CURRENCY_V6"
output_rows = []
selected["final_label"] = selected["raw_label"].astype(str)
rows_to_save = selected.sort_values(["final_label", "path_order"])
for row in rows_to_save.to_dict("records"):
    raw_bytes = archive.read(row["member"])
    with Image.open(io.BytesIO(raw_bytes)) as image:
        image = image.convert("RGB")
        crop_box = context_box(row["bbox"], image.width, image.height)
        crop = image.crop(crop_box)
    relative_path = f"train/{row['final_label']}/CURRENCY_{row['source_name']}.png"
    target = DATA_OUT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    crop.save(target, format="PNG", optimize=False)
    output_rows.append((row, raw_bytes, relative_path, crop_box, crop, target))
manifest_rows = []
for row, raw_bytes, relative_path, crop_box, crop, target in output_rows:
    manifest_rows.append({"relative_path": relative_path, "split": "train", "denomination": row["final_label"],
        "dataset_id": "currency_v6", "source_name": row["source_name"],
        "source_archive_member": row["member"], "annotation_member": row["annotation_member"],
        "raw_label": str(row["raw_label"]), "capture_group": row["capture_group"],
        "source_image_sha256": hashlib.sha256(raw_bytes).hexdigest(),
        "bbox_x1": row["bbox"][0], "bbox_y1": row["bbox"][1],
        "bbox_x2": row["bbox"][2], "bbox_y2": row["bbox"][3],
        "bbox_short_side_px": row["bbox_short_side_px"],
        "crop_x1": crop_box[0], "crop_y1": crop_box[1],
        "crop_x2": crop_box[2], "crop_y2": crop_box[3],
        "crop_width": crop.width, "crop_height": crop.height,
        "final_crop_sha256": hashlib.sha256(target.read_bytes()).hexdigest()})
result = pd.DataFrame(manifest_rows)
result.to_csv(DATA_OUT / "manifest.csv", index=False)
actual_counts = result.groupby("denomination").size().reindex(TARGET_CLASSES, fill_value=0).to_dict()
assert len(result) == 92
assert actual_counts == EXPECTED_COUNTS
print("Số ảnh cuối theo lớp:", actual_counts)
print("Tổng PNG:", len(list(DATA_OUT.rglob("*.png"))))


Số ảnh cuối theo lớp: {'10000': 19, '20000': 18, '50000': 9, '100000': 9, '200000': 19, '500000': 18}
Tổng PNG: 92


In [9]:
with ZipFile(OUTPUT / "CURRENCY_V6.zip", "w", ZIP_DEFLATED) as package:
    for path in sorted(DATA_OUT.rglob("*")):
        if path.is_file():
            package.write(path, path.relative_to(DATA_OUT).as_posix())
archive.close()
print("Ảnh crop:", DATA_OUT)
print("ZIP:", OUTPUT / "CURRENCY_V6.zip")

Ảnh crop: C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6\20261025_183559_632069\CURRENCY_V6
ZIP: C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6\20261025_183559_632069\CURRENCY_V6.zip
